# 04 · SFT Training: Qwen2.5-0.5B → function calling (QLoRA)

**Goal:** fine-tune on `data/sft/` and score the result **exactly** like the baseline (`03_baseline_eval`): same frozen benchmark, same prompt (`src/prompts.py`), same generator (`src/generation.py`) and same scorer (`src/evaluate.py`).

### How to run (one GPU run per day, so it has to work first time)
1. **`SMOKE_TEST = True` → Run All (~5–10 min).** A dry run of *every* cell: 10 steps on the **longest** examples (worst-case VRAM), save, merge, generate, score, save. It prints peak VRAM and an estimate of the full run time.
2. **`SMOKE_TEST = False` → Run All** and leave it. Smoke outputs go to separate `*_smoke` folders, so they never mix with real results.

### What's fixed compared with `Training_SFT_cleaned`
| Problem | Fix |
|---|---|
| `MAX_SEQ_LENGTH = 512` silently dropped 46% of the training data | read `1536` from `sft_data_report.json` (the data-prep decision, 99.5% coverage) + assert that **nothing** is dropped |
| inference prompt had no system prompt (Qwen inserts its own default) | eval uses `src.prompts.build_messages`, the same as the baseline and the training data |
| best checkpoint could be deleted | `load_best_model_at_end` on eval loss |
| a crash or sleeping laptop = lost day | resumes automatically from the last checkpoint |
| 4-bit `numel()` parameter counts were confusing | explained below and `print_trainable_parameters()` used instead |

## 0. Setup
No `transformers` / `unsloth` imports here on purpose. Unsloth has to be imported **before** them (it patches them), and that happens in Part A.

In [ ]:
import gc
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

# find the repo root (folder containing src/) and the data folder, wherever this notebook lives
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
DATA_DIR = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "splits").is_dir())
sys.path.insert(0, str(REPO_ROOT))

from src.data_utils import read_jsonl, verify_hashes                # noqa: E402
from src.prompts import PROMPT_VERSION, build_messages              # noqa: E402

print("Repo root:", REPO_ROOT)
print("Data dir: ", DATA_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none!")

## 1. Config
- **`SMOKE_TEST`**: the only line you change between the dry run and the real run.
- **`NUM_EPOCHS = 1`**: 48k examples of very templated data. One epoch is usually enough, and it fits the daily GPU window. If val loss is still falling at the end, try 2 epochs in a new run.
- **Effective batch = `BATCH_SIZE × GRAD_ACCUM` = 16.** Batch 2 keeps the longest 1536-token examples within 6 GB.
- **`MAX_SEQ_LENGTH` is not typed here.** It's read from the data-prep report, so training can never disagree with the data again. The file hashes are checked too.

In [ ]:
SMOKE_TEST = True          # True = 5-10 min dry run of everything. False = the real run.

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
RUN_NAME = "sft_qwen2.5-0.5b"
BENCHMARK = "test_v1"      # must match the baseline run
SEED = 3407

# LoRA
LORA_R = 16
LORA_ALPHA = 32

# training
NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8             # effective batch = 16
LEARNING_RATE = 2e-4
EVAL_EVERY = 200           # steps between eval-loss checks (= checkpoint interval)
SMOKE_STEPS = 10

# --- from the data-prep stage: never retype these ---
sft_report = json.loads((DATA_DIR / "sft" / "sft_data_report.json").read_text())
MAX_SEQ_LENGTH = sft_report["max_seq_len"]
RESPONSE_MARKER = sft_report["response_marker"]          # "<|im_start|>assistant\n"
assert sft_report["prompt_version"] == PROMPT_VERSION, "SFT data was built with a different prompt version"
verify_hashes(DATA_DIR / "sft", sft_report["files"])     # train/val.jsonl unchanged since data prep

# --- paths ---
if SMOKE_TEST:
    RUN_NAME += "_smoke"
OUTPUT_DIR = DATA_DIR.parent / "outputs" / RUN_NAME       # checkpoints (gitignored)
ADAPTER_DIR = OUTPUT_DIR / "adapter"                      # final LoRA adapter (small)
MERGED_DIR = OUTPUT_DIR / "merged"                        # base + adapter in bf16, used for eval and DPO
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = EVAL_DIR / "results" / RUN_NAME
BASE_RESULTS_DIR = EVAL_DIR / "results" / "base_qwen2.5-0.5b"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'SMOKE TEST' if SMOKE_TEST else 'REAL RUN'} -> {RUN_NAME}")
print(f"max_seq_length={MAX_SEQ_LENGTH} (from data prep) | prompt {PROMPT_VERSION} | data hashes ✅")

# Part A · Train

## 2. Load the model (4-bit) and add LoRA
**About parameter counts in 4-bit:** `bitsandbytes` packs two 4-bit weights into one byte, so `p.numel()` counts every quantized matrix at **half** its real size. That's why the old notebook showed 342M, 351M *and* 502M. `print_trainable_parameters()` corrects for the packing, so trust that one.

LoRA trains `r × (d_in + d_out)` parameters per layer instead of `d_in × d_out`. For a 896×896 `q_proj` with r=16 that's 28,672 instead of 802,816.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported   # first: patches transformers/trl

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,                      # auto -> bf16 on the RTX 4050
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,                # 0 = Unsloth's fast path
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

## 3. Data
- `.rstrip()` makes every example end **exactly** on `<|im_end|>`. With a trailing `\n`, TRL can decide the text "doesn't end with EOS" and append a second one.
- Only the `text` column is kept. With a `messages` column present, TRL may treat the dataset as a chat dataset and re-apply the template itself (a second formatting path we don't want).
- **Smoke test** uses the 160 **longest** examples, so the VRAM it measures is the worst case.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": str(DATA_DIR / "sft" / "train.jsonl"),
    "validation": str(DATA_DIR / "sft" / "val.jsonl"),
})
dataset = dataset.map(lambda x: {"text": x["text"].rstrip()})
dataset = dataset.remove_columns(["id", "messages"])

train_ds, val_ds = dataset["train"], dataset["validation"]
full_train_chars = sum(len(t) for t in train_ds["text"])        # used for the time estimate

if SMOKE_TEST:
    lengths = [len(t) for t in train_ds["text"]]
    longest = sorted(range(len(lengths)), key=lambda i: lengths[i], reverse=True)
    train_ds = train_ds.select(longest[: SMOKE_STEPS * BATCH_SIZE * GRAD_ACCUM])
    val_ds = val_ds.select(range(50))

print(f"train: {len(train_ds):,} | val: {len(val_ds):,}")
print(repr(train_ds[0]["text"][-80:]))

## 4. Trainer
- **bf16** because the model was loaded in bf16 (that was the original `TypeError`).
- **`train_on_responses_only`**: loss only on the JSON answer. The system prompt and tool list are ~90% of each example (`median_share_of_tokens_trained` = 9% in the data report), so without masking most of the gradient would go into memorizing boilerplate.
- **`load_best_model_at_end`**: the HF version of the early-stopping-on-val-loss you wrote by hand for SmolLM2. `save_total_limit=2` keeps the best checkpoint plus the latest one.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

eval_every = 5 if SMOKE_TEST else EVAL_EVERY

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,                 # < 1 means a ratio of total steps
    weight_decay=0.01,
    optim="adamw_8bit",

    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),

    logging_steps=1 if SMOKE_TEST else 10,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part=RESPONSE_MARKER,
)

## 5. Checks before spending GPU hours
If any of these fail, **stop**. They are the bugs that cost a whole training day:
1. **No example was dropped.** With 512 tokens, Unsloth dropped 22k.
2. **Only the answer is trained:** the unmasked tokens decode to exactly `answer + <|im_end|>`. That's the same check as in data prep, now run on what the trainer will *actually* use.

In [ ]:
assert len(trainer.train_dataset) == len(train_ds), \
    f"{len(train_ds) - len(trainer.train_dataset)} train examples dropped - is MAX_SEQ_LENGTH too small?"
assert len(trainer.eval_dataset) == len(val_ds), \
    f"{len(val_ds) - len(trainer.eval_dataset)} val examples dropped"

def trained_part(row):
    return tokenizer.decode([t for t, l in zip(row["input_ids"], row["labels"]) if l != -100])

for i in range(min(200, len(train_ds))):
    expected = train_ds[i]["text"].split(RESPONSE_MARKER)[-1]      # the JSON answer + <|im_end|>
    got = trained_part(trainer.train_dataset[i])
    assert got == expected, f"row {i}: trained on\n{got!r}\nexpected\n{expected!r}"

row = trainer.train_dataset[0]
n_trained = sum(l != -100 for l in row["labels"])
print(f"✅ nothing dropped ({len(trainer.train_dataset):,} train / {len(trainer.eval_dataset):,} val)")
print(f"✅ only the answer is trained, e.g. {n_trained} of {len(row['labels'])} tokens: {trained_part(row)!r}")

## 6. Train
Before the real run: **plug the laptop in and set Windows sleep to "Never"**. If it still crashes, re-run the notebook. It **resumes from the last checkpoint** (saved every 200 steps) instead of starting over.

The adapter is saved straight after training, so Part B can be re-run later without retraining.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None
print("Resuming from", last_checkpoint if last_checkpoint else "scratch")

torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
peak_train_vram = torch.cuda.max_memory_allocated() / 1024**3

model.save_pretrained(str(ADAPTER_DIR))         # best checkpoint (load_best_model_at_end)
tokenizer.save_pretrained(str(ADAPTER_DIR))
(RESULTS_DIR / "train_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))

print(f"\nTrain loss:      {train_result.training_loss:.4f}")
print(f"Best eval loss:  {trainer.state.best_metric:.4f}  ({trainer.state.best_model_checkpoint})")
print(f"Runtime:         {train_result.metrics['train_runtime'] / 3600:.2f} h")
print(f"Peak VRAM:       {peak_train_vram:.2f} GB of 6")
print(f"Adapter saved -> {ADAPTER_DIR}")

if SMOKE_TEST:
    smoke_chars = sum(len(t) for t in train_ds["text"])
    est_hours = train_result.metrics["train_runtime"] / smoke_chars * full_train_chars * NUM_EPOCHS / 3600
    print(f"\n⏱  Full run estimate: ~{est_hours:.1f} h of training (upper bound: smoke used the longest examples)")
    print("   + evaluation: about 45 min on test_v1 (the baseline took ~0.95 s/question)")
    if peak_train_vram > 5.5:
        print("⚠️  Peak VRAM is close to 6 GB. Set BATCH_SIZE=1, GRAD_ACCUM=16 for the real run.")

### Loss curves
What you want to see: eval loss falls and then flattens. If it **rises** while train loss keeps falling, that's overfitting. `load_best_model_at_end` already kept the best checkpoint, so you're protected either way.

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(9, 3.5))
log.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train loss", alpha=0.6)
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, label="eval loss", marker="o")
ax.set_title(f"{RUN_NAME}")
plt.tight_layout()
plt.show()

# Part B · Evaluate (identical to the baseline)
**Restarted the kernel, or the eval crashed?** Run sections 0 and 1, then continue from here. Part B only needs the saved adapter.

### 7. Build the eval model
The baseline was the **bf16** model loaded with plain `transformers`. To compare models and not loading methods, we do the same thing here: load the bf16 base, apply the adapter and **merge** it in. The merged model is saved, which is also the starting point for DPO later.

**Kernel check:** Unsloth patched `transformers` in this kernel during Part A. Before applying the adapter, the untouched base model must reproduce the saved baseline answers. If it does, this kernel generates the same way the baseline notebook did.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.evaluate import (breakdown, compare_runs, extract_json, failure_mix,      # noqa: E402
                          normalize_calls, query_template, score_generations, summarize, tag_features)
from src.generation import run_generation                                         # noqa: E402

# free the training model
trainer = model = None
gc.collect()
torch.cuda.empty_cache()

# frozen benchmark: refuse to run if it changed since the baseline
manifest = json.loads((EVAL_DIR / "manifest.json").read_text())
verify_hashes(EVAL_DIR, manifest["files"])
test_records = read_jsonl(EVAL_DIR / f"{BENCHMARK}.jsonl")
if SMOKE_TEST:
    test_records = test_records[:32]
records_by_id = {r["id"]: r for r in test_records}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype="auto", device_map="auto")
base_model.eval()

# --- kernel check: base model vs saved baseline answers ---
base_df_full = pd.read_csv(BASE_RESULTS_DIR / "scored.csv")
saved_base = base_df_full.set_index("id")["response"]
check_records = test_records[:16]
check_path = RESULTS_DIR / "kernel_check.jsonl"
check_path.unlink(missing_ok=True)
check_out = run_generation(base_model, tokenizer, check_records, build_messages, out_path=check_path,
                           run_config={"check": "base"}, batch_size=16, max_new_tokens=512)
same = sum(check_out[r["id"]]["response"] == saved_base[r["id"]] for r in check_records)
print(f"Kernel check: {same}/{len(check_records)} base answers identical to the baseline run")
if same < len(check_records) - 2:
    print("⚠️  Generation differs in this kernel. Restart the kernel, run sections 0-1, then Part B.")

# --- apply adapter and merge ---
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR)).merge_and_unload()
model.eval()
model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Merged bf16 model saved -> {MERGED_DIR}")

### 8. Generate on the frozen benchmark
Same benchmark, prompt version, greedy decoding and `max_new_tokens` as the baseline. Resumable: if this cell is interrupted, re-run it and it continues where it stopped.

In [ ]:
MAX_NEW_TOKENS = 512
RUN_CONFIG = {
    "model": f"{MODEL_NAME} + {RUN_NAME} (merged LoRA)", "benchmark": BENCHMARK,
    "benchmark_sha256": manifest["files"][f"{BENCHMARK}.jsonl"]["sha256"],
    "prompt_version": PROMPT_VERSION, "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy",
}

torch.cuda.reset_peak_memory_stats()
generations = run_generation(
    model, tokenizer, test_records, build_messages,
    out_path=RESULTS_DIR / "predictions.jsonl", run_config=RUN_CONFIG,
    batch_size=16, max_new_tokens=MAX_NEW_TOKENS,
)
peak_eval_vram = torch.cuda.max_memory_allocated() / 1024**3

### 9. Base vs SFT, question by question
- **Summary:** both runs scored on the same questions (in smoke mode the base numbers are recomputed on the same 32).
- **`compare_runs`:** `fixed` = questions SFT gets right that base got wrong, `broke` = the reverse. A McNemar p < 0.05 means the improvement is real and not noise.

In [ ]:
train_templates = {query_template(r["query"]) for r in read_jsonl(DATA_DIR / "splits" / "sft_train.jsonl")}
features_df = pd.DataFrame([{"id": r["id"], **tag_features(r, train_templates)} for r in test_records])

sft_df = score_generations(generations, records_by_id, features_df)
assert len(sft_df) == len(test_records), "some questions have no prediction"
base_df = base_df_full[base_df_full["id"].isin(sft_df["id"])]

sft_summary = summarize(sft_df)
base_summary = summarize(base_df)
summary_table = pd.DataFrame({
    "base": base_summary["value"],
    "sft": sft_summary["value"],
    "change": (sft_summary["value"] - base_summary["value"]).round(1),
    "sft 95% CI": sft_summary["ci_low"].astype(str) + " – " + sft_summary["ci_high"].astype(str),
})
display(summary_table)

comparison = compare_runs(base_df, sft_df)
comparison

### Where did it improve?
The row to watch is **`template_twin`**. Gains on `novel` questions mean real generalisation. Gains only on `seen_template` mean it mostly memorised patterns from training.

In [ ]:
FEATURES = ["call_pattern", "calls_bucket", "tools_bucket", "arg_complexity", "has_optional", "template_twin"]
for feature in FEATURES:
    side_by_side = breakdown(base_df, feature).rename(columns={"exact_match": "base"}).join(
        breakdown(sft_df, feature)["exact_match"].rename("sft"))
    side_by_side["change"] = (side_by_side["sft"] - side_by_side["base"]).round(1)
    print(f"\n=== {feature} ===")
    display(side_by_side)

display(pd.DataFrame({"base %": failure_mix(base_df), "sft %": failure_mix(sft_df)}).fillna(0))

In [ ]:
# eyeball one remaining failure of each type, so you can tell what DPO should target
for ftype, group in sft_df[sft_df["failure_type"] != "correct"].groupby("failure_type"):
    rec = records_by_id[group.iloc[0]["id"]]
    print(f"##### {ftype}  ({len(group)} questions)")
    print("QUERY :", rec["query"])
    print("GOLD  :", json.dumps(rec["answers"], ensure_ascii=False)[:300])
    print("MODEL :", str(group.iloc[0]["response"])[:300], "\n")

### 10. General-behaviour check (regression)
The same 20 prompts as the baseline. Expect plain chat to stay at ~0% tool-call JSON. **"No tool fits → `[]`" will probably not improve, and may drop:** xLAM has no examples where the right answer is `[]`. That's a known gap for the DPO stage, not a bug.

In [ ]:
general = read_jsonl(EVAL_DIR / "general_prompts_v1.jsonl")
chat_prompts = [g for g in general if g["kind"] == "chat"]
irrelevant_prompts = [g for g in general if g["kind"] == "irrelevant_tools"]

chat_out = run_generation(
    model, tokenizer, chat_prompts, lambda r: [{"role": "user", "content": r["query"]}],
    out_path=RESULTS_DIR / "general_chat.jsonl", run_config={**RUN_CONFIG, "set": "chat"},
    batch_size=16, max_new_tokens=200, stop_at_json=False,
)
irrelevant_out = run_generation(
    model, tokenizer, irrelevant_prompts, build_messages,
    out_path=RESULTS_DIR / "general_irrelevant.jsonl", run_config={**RUN_CONFIG, "set": "irrelevant"},
    batch_size=16, max_new_tokens=MAX_NEW_TOKENS,
)

chat_tool_rate = 100 * sum(bool(normalize_calls(extract_json(chat_out[g["id"]]["response"]))) for g in chat_prompts) / len(chat_prompts)
irrelevance_acc = 100 * sum(normalize_calls(extract_json(irrelevant_out[g["id"]]["response"])) == [] for g in irrelevant_prompts) / len(irrelevant_prompts)

base_general = json.loads((BASE_RESULTS_DIR / "metrics.json").read_text())["general_behaviour"]
print(f"Plain chat answered as tool-call JSON: base {base_general['chat_as_tool_call_pct']:.0f}% -> sft {chat_tool_rate:.0f}%  (lower is better)")
print(f"'No tool fits' correctly returned []:  base {base_general['irrelevance_returns_empty_pct']:.0f}% -> sft {irrelevance_acc:.0f}%  (higher is better)")

with pd.option_context("display.max_colwidth", 150):
    display(pd.DataFrame([{"query": g["query"], "sft response": chat_out[g["id"]]["response"][:150]} for g in chat_prompts]))

## 11. Save
Same files and `metrics.json` layout as the baseline, plus the training info. DPO will compare against `scored.csv` from here.

In [ ]:
sft_df.to_csv(RESULTS_DIR / "scored.csv", index=False)

metrics_report = {
    "run_name": RUN_NAME,
    "run_config": RUN_CONFIG,
    "n": len(sft_df),
    "summary": sft_summary.to_dict(orient="index"),
    "vs_base": comparison,
    "breakdowns": {f: breakdown(sft_df, f).to_dict(orient="index") for f in FEATURES},
    "failure_mix": failure_mix(sft_df).to_dict(),
    "generation": {
        "avg_gen_tokens": round(sft_df["gen_tokens"].mean(), 1),
        "hit_max_tokens_pct": round(100 * sft_df["hit_max_tokens"].mean(), 1),
        "latency_per_example_s": round(sft_df["latency_per_example"].mean(), 3),
        "peak_vram_gb": round(peak_eval_vram, 2),
    },
    "general_behaviour": {"chat_as_tool_call_pct": chat_tool_rate, "irrelevance_returns_empty_pct": irrelevance_acc},
    "training": {
        "model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "epochs": NUM_EPOCHS, "effective_batch": BATCH_SIZE * GRAD_ACCUM, "learning_rate": LEARNING_RATE,
        "sft_data_files": sft_report["files"], "adapter_dir": str(ADAPTER_DIR), "merged_dir": str(MERGED_DIR),
    },
}
(RESULTS_DIR / "metrics.json").write_text(json.dumps(metrics_report, indent=2, default=str))
print(f"Saved to {RESULTS_DIR}")
if SMOKE_TEST:
    print("\n✅ Smoke test passed end-to-end. Set SMOKE_TEST = False and Run All for the real run.")